# Battery storage in MaStR — starter notebook

Pulls storage units (`Energieträger = Speicher`) for one Bundesland from the public MaStR JSON API, caches them as Parquet in `data/raw/`, and shows a few first cuts. Copy this notebook as a template for PV / CHP analyses.

Notebooks are numbered `NN_topic.ipynb`; reusable code goes into `mastr_analysis/`, not into notebooks.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from mastr_analysis import fetch_power_generation, load_parquet, save_parquet
from mastr_analysis.config import RAW_DIR, TECH_STORAGE
from mastr_analysis.io import parse_dates

pd.set_option("display.max_columns", 60)

BUNDESLAND = "Berlin"
DATASET = f"batteries_{BUNDESLAND.lower()}"

## 1. Fetch (or load cached)

In [ ]:
path = RAW_DIR / f"{DATASET}.parquet"
if path.exists():
    df = load_parquet(DATASET)
else:
    df = fetch_power_generation({"tech": TECH_STORAGE, "bundesland": BUNDESLAND})
    save_parquet(df, DATASET)
    df = parse_dates(df)

print(df.shape)
df.head()

## 2. Key columns

| Column | Meaning |
|---|---|
| `MaStRNummer` | Unit id (SEE…) |
| `Bruttoleistung` / `Nettonennleistung` | Gross / net power in kW |
| `NutzbareSpeicherkapazitaet` | Usable storage capacity in kWh |
| `Batterietechnologie` | Dropdown id (727 = Lithium) |
| `BetriebsStatusName` | In Betrieb / In Planung / Endgültig stillgelegt |
| `InbetriebnahmeDatum` | Commissioning date |
| `AnlagenbetreiberName`, `NetzbetreiberNamen` | Operator / DSO |
| `Plz`, `Ort`, `Landkreis`, `Breitengrad`, `Laengengrad` | Location |

In [ ]:
df["BetriebsStatusName"].value_counts()

## 3. Capacity distribution and commissioning over time

In [ ]:
active = df[df["BetriebsStatusName"] == "In Betrieb"].copy()

bins = [0, 10, 30, 100, 1000, 10_000, float("inf")]
labels = ["<10 kW", "10–30 kW", "30–100 kW", "100 kW–1 MW", "1–10 MW", ">10 MW"]
active["size_class"] = pd.cut(active["Bruttoleistung"], bins=bins, labels=labels)

active.groupby("size_class", observed=True).agg(
    units=("MaStRNummer", "count"),
    power_mw=("Bruttoleistung", lambda s: s.sum() / 1000),
)

In [ ]:
yearly = (
    active.dropna(subset=["InbetriebnahmeDatum"])
    .assign(year=lambda d: d["InbetriebnahmeDatum"].dt.year)
    .groupby("year")
    .agg(units=("MaStRNummer", "count"), power_mw=("Bruttoleistung", lambda s: s.sum() / 1000))
)
yearly.plot.bar(subplots=True, figsize=(10, 6), legend=False, title=f"Battery storage commissioned per year — {BUNDESLAND}");

## 4. Largest operators

In [ ]:
(
    active.groupby("AnlagenbetreiberName")
    .agg(units=("MaStRNummer", "count"), power_mw=("Bruttoleistung", lambda s: s.sum() / 1000))
    .sort_values("power_mw", ascending=False)
    .head(15)
)